In [1]:
import pandas as pd
import numpy as np

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
import sqlite3

conn = sqlite3.connect("../db.sqlite3")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

tables = cursor.fetchall()
[print(tab) for tab in tables]



In [ ]:
query = f"SELECT * FROM {tables[1][0]}"

df = pd.read_sql_query(query, conn)
df


In [ ]:
query = f"SELECT * FROM {tables[2][0]}"

df_products = pd.read_sql_query(query, conn)
df_products = df_products.rename(columns={"id": "product_id"})
df_products

In [ ]:
df = df.merge(df_products[["product_id", "price", "name"]], on="product_id", how="left")
df

In [ ]:
query = f"SELECT * FROM {tables[3][0]}"

df_orders = pd.read_sql_query(query, conn, parse_dates=["datetime"])
df_orders.rename(columns={"id": "order_id", "datetime": "order_datetime"}, inplace=True)
df_orders.info()

In [ ]:
df_orders

In [ ]:
df = df.merge(df_orders[["order_id", "order_datetime"]], on="order_id", how="left")
df

In [236]:
df.shape[0]

74818

In [238]:
df.to_csv("../data/db_file.csv")

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
popular_products = df.groupby("name")["quantity"].sum("Quantity").sort_values(ascending=False).head(10)
popular_products


In [ ]:
import matplotlib.pyplot as plt

def autopct_format(values):
    def inner(pct):
        total_sum = sum(values)
        val = int(round(total_sum / 100 * pct))
        return f"{pct:.1f}% ({val})"
    return inner

popular_products.plot.pie(figsize=(10, 8), autopct=autopct_format(popular_products.values), startangle=75)
plt.title("Top 10 positions in menu by quantity")
plt.ylabel("Quantity")
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["item_price"] = df["price"] * df["quantity"]
df.head(10)

In [ ]:
profitable_products = df.groupby("name")["item_price"].sum().sort_values(ascending=False).head(10)
profitable_products

In [ ]:
profitable_products.plot.pie(figsize=(10, 8), autopct=autopct_format(profitable_products.values), startangle=70)
plt.title("Top 10 positions in menu by profit")
plt.ylabel("Income")
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df["order_hour"] = df["order_datetime"].dt.hour
df

In [ ]:
orders_by_hours = df.groupby("order_hour")["item_price"].sum()
orders_by_hours

In [ ]:
orders_by_hours.plot.bar(figsize=(14, 8))
plt.title("Profit by Order Hour")
plt.xlabel("Order Hour")
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df["day_week"] = df["order_datetime"].dt.day_name()
df

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

orders_by_day_of_week = df.groupby("day_week")["item_price"].sum().reindex(weekday_order)
orders_by_day_of_week

In [ ]:
orders_by_day_of_week.plot.bar(figsize=(10, 6))
plt.title("Profit by Day of The Week")
plt.xlabel("Day of The Week")
plt.xticks(rotation=45)
plt.show()